# RICES embedding construction — MMMU demonstrations × MMMU-Pro Eval300 queries

This notebook constructs a frozen, retrieval-ready embedding corpus for later
few-shot **RICES** selection. It does **not** run retrieval and does **not**
build few-shot prompts.

## Inputs

Attach one Kaggle Dataset containing:

1. `selected_ids.txt` — the fixed ordered 300-ID MMMU-Pro query cohort.
2. `cot_pipeline_clean_merged_KEEP_only_grouped_by_subject.csv` — the frozen
   405-ID demonstration selection.

Internet must be enabled. A GPU is recommended; no API key is required for the
public Hugging Face repositories.

## Frozen semantic inputs

For every selected ID, the notebook downloads and reconstructs the source row
from a full immutable Hugging Face commit:

- queries: `MMMU/MMMU_Pro`, physical setting `standard (10 options)`, split
  `test`, revision `563f3e84bb3b90893083a1f039cfa13077f2302b`;
- demonstrations: `MMMU/MMMU`, revision
  `98e6ac0cb9b7b2cd2c991b85a50762edc4aedc68`.

The CSV is used only to select and locate demonstration IDs. Its option,
answer, explanation, and generated-CoT fields are never used as embedding
input.

## Embedding protocol

- Encoder: `openai/clip-vit-large-patch14`, pinned to full commit
  `32bd64288804d66eefd0ccbe215aa642df71cc41`.
- Text input: raw question stem only, after removing literal `<image n>`
  placeholders and normalizing whitespace. If no text remains, the row is
  explicitly marked text-missing and receives a zero vector so that its later
  text-similarity contribution is exactly zero.
- Image input: each physical `image_1` … `image_7` field independently.
- Excluded: options, labels, gold answers, source explanations, generated CoT.
- Output: float32 embeddings. Available modalities are L2-normalized; a truly
  absent text modality is represented by a flagged zero vector. Dot product is
  therefore cosine similarity for present modalities and zero for missing text.
- Multi-image samples are **not aggregated**. Duplicate marker occurrences do
  not cause duplicate embeddings; their occurrence counts and positions are
  retained as metadata.

The four canonical Parquet datasets are:

1. `query_question_embeddings.parquet`
2. `query_image_embeddings.parquet`
3. `demonstration_question_embeddings.parquet`
4. `demonstration_image_embeddings.parquet`

Equivalent `.npz` matrices, audit manifests, source hashes, and a paper-ready
Methods paragraph are also produced.

## References

- [What Makes Multimodal In-Context Learning Work?](https://arxiv.org/html/2404.15736v1)
  — the RICES analysis and the reported `openai/clip-vit-large-patch14` encoder.
- [Official CLIP ViT-L/14 model](https://huggingface.co/openai/clip-vit-large-patch14)
- [Official MMMU dataset](https://huggingface.co/datasets/MMMU/MMMU)
- [Official MMMU-Pro dataset](https://huggingface.co/datasets/MMMU/MMMU_Pro)

For the later score

$$S_{iq}=s(I_i,I_q)+s(T_i,T_q),$$

this project defines $T$ as the **question stem only**, exactly as requested.
This is a declared project-specific variant: the cited paper describes
question–answer text for its VQA experiments. No answer text is allowed into
the present embedding corpus, so answer content cannot influence retrieval.


In [1]:
# 0. Install the exact user-space libraries used by this protocol.
# PyTorch is intentionally not reinstalled because Kaggle supplies a CUDA-matched build.
import subprocess, sys

PINS = [
    "transformers==4.53.3",
    "datasets==3.6.0",
    "huggingface_hub==0.33.4",
    "safetensors==0.5.3",
]
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", *PINS],
    check=True,
)
print("Pinned Python packages installed:", PINS)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 44.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.3/515.3 kB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 471.6/471.6 kB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 98.8 MB/s eta 0:00:00
Pinned Python packages installed: ['transformers==4.53.3', 'datasets==3.6.0', 'huggingface_hub==0.33.4', 'safetensors==0.5.3']


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tpot 1.1.0 requires dill>=0.3.9, but you have dill 0.3.8 which is incompatible.
diffusers 0.37.1 requires huggingface-hub<2.0,>=0.34.0, but you have huggingface-hub 0.33.4 which is incompatible.
gradio 5.50.0 requires huggingface-hub<2.0,>=0.33.5, but you have huggingface-hub 0.33.4 which is incompatible.


In [2]:
# 1. Imports, deterministic execution, immutable protocol constants
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import io, re, json, hashlib, random, platform, sys, subprocess, tempfile
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from PIL import Image, ImageOps
from tqdm.auto import tqdm

import torch
import transformers, datasets, huggingface_hub, safetensors
from datasets import load_dataset, Image as HFImage
from huggingface_hub import snapshot_download
from transformers import CLIPModel, CLIPProcessor

PROTOCOL_VERSION = "RICES_MMMU_MMMUPRO_CLIP_VITL14_EMBEDDINGS_V1_1_20260905"

QUERY_REPO = "MMMU/MMMU_Pro"
QUERY_SUBDIR = "standard (10 options)"
QUERY_SPLIT = "test"
QUERY_REVISION = "563f3e84bb3b90893083a1f039cfa13077f2302b"
QUERY_EXPECTED_SOURCE_ROWS = 1730
QUERY_EXPECTED_ROWS = 300
QUERY_IDS_CANONICAL_SHA256 = "db7ec6dca5dff71d8ea8551a08c448c4314e153509ce9c78d2e8c652011a5dc3"

DEMO_REPO = "MMMU/MMMU"
DEMO_REVISION = "98e6ac0cb9b7b2cd2c991b85a50762edc4aedc68"
DEMO_EXPECTED_ROWS = 405
DEMO_CSV_SHA256 = "2c55994efcbaf2d1e21e43f256a70e3e10859f6f20eafb68c41a48d9ba2666ca"
DEMO_IDS_CANONICAL_SHA256 = "92bcccab502feaf63e61cd3c49cae591cd41b3c310a8eae0cccef8a814c2a635"

CLIP_REPO = "openai/clip-vit-large-patch14"
CLIP_REVISION = "32bd64288804d66eefd0ccbe215aa642df71cc41"
CLIP_EXPECTED_MODEL_SHA256 = "a2bf730a0c7debf160f7a6b50b3aaf3703e7e88ac73de7a314903141db026dcb"
CLIP_EXPECTED_DIM = 768
CLIP_EXPECTED_CONTEXT = 77

IMAGE_COLS = [f"image_{i}" for i in range(1, 8)]
IMAGE_REF_RE = re.compile(r"<image\s*([1-7])\s*>", flags=re.IGNORECASE)
WHITESPACE_RE = re.compile(r"\s+")
TEXT_NORMALIZATION_POLICY = (
    "remove literal <image n> markers; Unicode preserved; collapse all whitespace; strip"
)
IMAGE_PREPROCESS_POLICY = (
    "decode official source bytes; apply EXIF orientation; convert RGB; "
    "CLIPProcessor at pinned model revision"
)
EMBEDDING_POLICY = (
    "float32 inference; unit-L2 for present modalities; zero vector for absent text"
)
EMPTY_TEXT_POLICY = (
    "if marker removal leaves no text: store a 768-dimensional float32 zero vector; "
    "set question_has_text_after_marker_removal=false and text_similarity_eligible=false"
)

INPUT_ROOT = Path("/kaggle/input")
OUTPUT_ROOT = Path("/kaggle/working/RICES_MMMU_MMMUPro_CLIP_ViTL14_embeddings_v1_1")
QUERY_CACHE = Path("/kaggle/working/source_cache_mmmu_pro_563f3e84")
DEMO_CACHE = Path("/kaggle/working/source_cache_mmmu_98e6ac0c")
MODEL_CACHE = Path("/kaggle/working/model_cache_clip_vitl14_32bd642")
for directory in (OUTPUT_ROOT, QUERY_CACHE, DEMO_CACHE, MODEL_CACHE):
    directory.mkdir(parents=True, exist_ok=True)

if not INPUT_ROOT.exists():
    raise RuntimeError("/kaggle/input is missing; this notebook is intended for Kaggle.")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.use_deterministic_algorithms(True, warn_only=False)
torch.set_float32_matmul_precision("highest")

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
TEXT_BATCH_SIZE = 64
IMAGE_BATCH_SIZE = 16 if DEVICE.type == "cuda" else 4


def utc_now():
    return datetime.now(timezone.utc).isoformat()


def sha256_bytes(data):
    return hashlib.sha256(data).hexdigest()


def sha256_file(path, chunk_size=8 * 1024 * 1024):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


def canonical_id_text(ids):
    return "\n".join(str(x) for x in ids) + "\n"


def canonical_id_sha256(ids):
    return sha256_bytes(canonical_id_text(ids).encode("utf-8"))


def require(condition, message):
    if not condition:
        raise RuntimeError(message)


def atomic_json(path, payload):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
    os.replace(tmp, path)


def environment_snapshot():
    gpu_names = []
    if torch.cuda.is_available():
        gpu_names = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
    return {
        "captured_utc": utc_now(),
        "protocol_version": PROTOCOL_VERSION,
        "python": sys.version,
        "platform": platform.platform(),
        "torch": torch.__version__,
        "cuda_runtime": torch.version.cuda,
        "cudnn": torch.backends.cudnn.version(),
        "transformers": transformers.__version__,
        "datasets": datasets.__version__,
        "huggingface_hub": huggingface_hub.__version__,
        "safetensors": safetensors.__version__,
        "pandas": pd.__version__,
        "numpy": np.__version__,
        "pyarrow": pa.__version__,
        "pillow": Image.__version__,
        "device": str(DEVICE),
        "gpu_names": gpu_names,
        "text_batch_size": TEXT_BATCH_SIZE,
        "image_batch_size": IMAGE_BATCH_SIZE,
        "seed": SEED,
        "deterministic_algorithms": True,
        "float32_matmul_precision": "highest",
        "tf32_allowed": False,
    }


ENVIRONMENT = environment_snapshot()
atomic_json(OUTPUT_ROOT / "environment.json", ENVIRONMENT)
subprocess.run(
    [sys.executable, "-m", "pip", "freeze"],
    check=True,
    stdout=(OUTPUT_ROOT / "pip_freeze.txt").open("w", encoding="utf-8"),
)
print(json.dumps(ENVIRONMENT, indent=2))


{
  "captured_utc": "2026-09-05T09:54:49.577797+00:00",
  "protocol_version": "RICES_MMMU_MMMUPRO_CLIP_VITL14_EMBEDDINGS_V1_1_20260905",
  "python": "3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]",
  "platform": "Linux-6.12.90+-x86_64-with-glibc2.35",
  "torch": "2.10.0+cu128",
  "cuda_runtime": "12.8",
  "cudnn": 91002,
  "transformers": "4.53.3",
  "datasets": "3.6.0",
  "huggingface_hub": "0.33.4",
  "safetensors": "0.5.3",
  "pandas": "2.3.3",
  "numpy": "2.0.2",
  "pyarrow": "24.0.0",
  "pillow": "11.3.0",
  "device": "cuda:0",
  "gpu_names": [
    "Tesla T4",
    "Tesla T4"
  ],
  "text_batch_size": 64,
  "image_batch_size": 16,
  "seed": 42,
  "deterministic_algorithms": true,
  "float32_matmul_precision": "highest",
  "tf32_allowed": false
}


In [3]:
# 2. Discover and cryptographically validate the two selection inputs
DEMO_INPUT_NAME = "cot_pipeline_clean_merged_KEEP_only_grouped_by_subject.csv"
QUERY_INPUT_NAME = "selected_ids.txt"


def discover_copies(filename):
    return sorted({p.resolve() for p in INPUT_ROOT.rglob(filename) if p.is_file()})


demo_candidates = discover_copies(DEMO_INPUT_NAME)
query_candidates = discover_copies(QUERY_INPUT_NAME)
require(demo_candidates, f"Missing Kaggle input: {DEMO_INPUT_NAME}")
require(query_candidates, f"Missing Kaggle input: {QUERY_INPUT_NAME}")

matching_demo = [p for p in demo_candidates if sha256_file(p) == DEMO_CSV_SHA256]
require(
    matching_demo,
    "No demonstration CSV matches the frozen SHA-256. "
    f"Expected={DEMO_CSV_SHA256}; discovered="
    + json.dumps({str(p): sha256_file(p) for p in demo_candidates}, indent=2),
)
DEMO_CSV_PATH = matching_demo[0]


def read_ids_strict(path):
    raw = Path(path).read_text(encoding="utf-8")
    lines = raw.splitlines()
    require(lines, f"Empty ID file: {path}")
    require(
        all(line and line == line.strip() for line in lines),
        f"Blank or whitespace-padded ID in {path}",
    )
    return lines


matching_query = []
for path in query_candidates:
    ids_here = read_ids_strict(path)
    if canonical_id_sha256(ids_here) == QUERY_IDS_CANONICAL_SHA256:
        matching_query.append((path, ids_here))
require(
    matching_query,
    "No selected_ids.txt matches the frozen canonical SHA-256 "
    f"{QUERY_IDS_CANONICAL_SHA256}",
)
QUERY_IDS_PATH, query_ids = matching_query[0]

require(len(query_ids) == QUERY_EXPECTED_ROWS, f"Expected 300 query IDs; got {len(query_ids)}")
require(len(set(query_ids)) == len(query_ids), "Query IDs are not unique")
require(canonical_id_sha256(query_ids) == QUERY_IDS_CANONICAL_SHA256, "Query-ID hash mismatch")

demo_selection = pd.read_csv(DEMO_CSV_PATH, low_memory=False)
required_demo_columns = {
    "question_id", "subject", "source_split", "hf_source_file",
    "hf_row_group_idx", "hf_local_row_in_group", "hf_global_split_offset",
    "image_sha256_json", "hf_image_bytes", "question", "dataset_revision",
    "accepted", "pipeline_status", "manual_logic_audit_recommendation",
}
missing = required_demo_columns - set(demo_selection.columns)
require(not missing, f"Demonstration CSV missing columns: {sorted(missing)}")

demo_ids = demo_selection["question_id"].astype(str).tolist()
require(len(demo_ids) == DEMO_EXPECTED_ROWS, f"Expected 405 demonstrations; got {len(demo_ids)}")
require(len(set(demo_ids)) == len(demo_ids), "Demonstration IDs are not unique")
require(canonical_id_sha256(demo_ids) == DEMO_IDS_CANONICAL_SHA256, "Demo-ID order/hash mismatch")
require(set(demo_selection["dataset_revision"].astype(str)) == {DEMO_REVISION}, "Demo revision mismatch")
require(demo_selection["accepted"].fillna(False).astype(bool).all(), "CSV contains unaccepted demos")
require(set(demo_selection["pipeline_status"].astype(str)) == {"ACCEPTED"}, "Non-ACCEPTED demo")
require(
    set(demo_selection["manual_logic_audit_recommendation"].astype(str)) == {"KEEP"},
    "Demo CSV contains a row not marked KEEP",
)
require(not set(query_ids).intersection(demo_ids), "Query/demo ID overlap detected")

input_audit = {
    "captured_utc": utc_now(),
    "query_ids_path": str(QUERY_IDS_PATH),
    "query_matching_copies": [str(x[0]) for x in matching_query],
    "query_rows": len(query_ids),
    "query_unique_ids": len(set(query_ids)),
    "query_ids_canonical_sha256": canonical_id_sha256(query_ids),
    "demo_csv_path": str(DEMO_CSV_PATH),
    "demo_matching_copies": [str(x) for x in matching_demo],
    "demo_csv_sha256": sha256_file(DEMO_CSV_PATH),
    "demo_rows": len(demo_ids),
    "demo_unique_ids": len(set(demo_ids)),
    "demo_ids_canonical_sha256": canonical_id_sha256(demo_ids),
    "query_demo_id_overlap": 0,
}
atomic_json(OUTPUT_ROOT / "selection_input_audit.json", input_audit)
(OUTPUT_ROOT / "selected_ids.txt").write_text(canonical_id_text(query_ids), encoding="utf-8")
demo_selection[["question_id", "subject", "source_split"]].to_csv(
    OUTPUT_ROOT / "demonstration_ids.csv", index=False
)
print(json.dumps(input_audit, indent=2))


{
  "captured_utc": "2026-09-05T09:54:51.571351+00:00",
  "query_ids_path": "/kaggle/input/datasets/jingilifteyna/300-mmmu-pro/selected_ids.txt",
  "query_matching_copies": [
    "/kaggle/input/datasets/jingilifteyna/300-mmmu-pro/selected_ids.txt"
  ],
  "query_rows": 300,
  "query_unique_ids": 300,
  "query_ids_canonical_sha256": "db7ec6dca5dff71d8ea8551a08c448c4314e153509ce9c78d2e8c652011a5dc3",
  "demo_csv_path": "/kaggle/input/datasets/jingilifteyna/pooling/cot_pipeline_clean_merged_KEEP_only_grouped_by_subject.csv",
  "demo_matching_copies": [
    "/kaggle/input/datasets/jingilifteyna/pooling/cot_pipeline_clean_merged_KEEP_only_grouped_by_subject.csv"
  ],
  "demo_csv_sha256": "2c55994efcbaf2d1e21e43f256a70e3e10859f6f20eafb68c41a48d9ba2666ca",
  "demo_rows": 405,
  "demo_unique_ids": 405,
  "demo_ids_canonical_sha256": "92bcccab502feaf63e61cd3c49cae591cd41b3c310a8eae0cccef8a814c2a635",
  "query_demo_id_overlap": 0
}


In [4]:
# 3. Download the pinned MMMU-Pro source and reconstruct the 300 queries by exact ID
print("Downloading pinned MMMU-Pro query source...")
query_snapshot = Path(snapshot_download(
    repo_id=QUERY_REPO,
    repo_type="dataset",
    revision=QUERY_REVISION,
    allow_patterns=[f"{QUERY_SUBDIR}/*.parquet"],
    local_dir=str(QUERY_CACHE),
))
query_shards = sorted((query_snapshot / QUERY_SUBDIR).glob(f"{QUERY_SPLIT}-*.parquet"))
require(len(query_shards) == 2, f"Expected exactly two query Parquet shards; got {query_shards}")

query_shard_manifest = [
    {
        "relative_path": str(path.relative_to(query_snapshot)),
        "size_bytes": path.stat().st_size,
        "sha256": sha256_file(path),
    }
    for path in query_shards
]

query_source = load_dataset(
    "parquet",
    data_files={QUERY_SPLIT: [str(p) for p in query_shards]},
    split=QUERY_SPLIT,
)
for image_col in IMAGE_COLS:
    require(image_col in query_source.column_names, f"Query source missing {image_col}")
    query_source = query_source.cast_column(image_col, HFImage(decode=False))

required_query_columns = {"id", "question", "subject", *IMAGE_COLS}
require(not (required_query_columns - set(query_source.column_names)), "Query source schema mismatch")
require(len(query_source) == QUERY_EXPECTED_SOURCE_ROWS, f"Query source rows={len(query_source)}")

query_source_ids = [str(x) for x in query_source["id"]]
require(len(set(query_source_ids)) == len(query_source_ids), "MMMU-Pro source IDs are not unique")
query_index_by_id = {sample_id: idx for idx, sample_id in enumerate(query_source_ids)}
missing_query_ids = [sample_id for sample_id in query_ids if sample_id not in query_index_by_id]
require(not missing_query_ids, f"Missing selected query IDs: {missing_query_ids[:20]}")

query_source_indices = [query_index_by_id[sample_id] for sample_id in query_ids]
query_subset = query_source.select(query_source_indices)
require([str(x) for x in query_subset["id"]] == query_ids, "Query reconstruction order mismatch")

query_records = []
for order0, (source_index, row) in enumerate(zip(query_source_indices, query_subset), start=1):
    query_records.append({
        "corpus": "query",
        "sample_id": str(row["id"]),
        "sample_order": order0,
        "subject": str(row["subject"]),
        "source_split": QUERY_SPLIT,
        "id_prefix_split": str(row["id"]).split("_", 1)[0],
        "source_repo": QUERY_REPO,
        "source_revision": QUERY_REVISION,
        "source_locator": f"source_row_index={source_index}",
        "source_row_index": int(source_index),
        "question": str(row["question"]),
        "source_row": dict(row),
    })

atomic_json(OUTPUT_ROOT / "query_source_provenance.json", {
    "captured_utc": utc_now(),
    "repo": QUERY_REPO,
    "physical_setting": QUERY_SUBDIR,
    "split": QUERY_SPLIT,
    "revision": QUERY_REVISION,
    "source_rows": len(query_source),
    "selected_rows": len(query_records),
    "selected_ids_canonical_sha256": canonical_id_sha256([r["sample_id"] for r in query_records]),
    "source_shards": query_shard_manifest,
})
print("MMMU-Pro exact-ID reconstruction: PASS", len(query_records))


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

standard (10 options)/test-00001-of-0000(…):   0%|          | 0.00/332M [00:00<?, ?B/s]

standard (10 options)/test-00000-of-0000(…):   0%|          | 0.00/346M [00:00<?, ?B/s]

Generating test split: 0 examples [00:00, ? examples/s]

MMMU-Pro exact-ID reconstruction: PASS 300


In [5]:
# 4. Download only the pinned MMMU Parquet files needed by the 405 demonstration IDs
demo_source_files = sorted(set(demo_selection["hf_source_file"].astype(str)))
require(all(path.endswith(".parquet") for path in demo_source_files), "Invalid demo source file path")
print(f"Downloading {len(demo_source_files)} exact MMMU source files at the pinned revision...")

demo_snapshot = Path(snapshot_download(
    repo_id=DEMO_REPO,
    repo_type="dataset",
    revision=DEMO_REVISION,
    allow_patterns=demo_source_files,
    local_dir=str(DEMO_CACHE),
))
for rel in demo_source_files:
    require((demo_snapshot / rel).is_file(), f"Missing pinned demo source file: {rel}")


def bytes_from_image_cell(value, source_dir=None):
    if value is None:
        return None
    if isinstance(value, memoryview):
        return value.tobytes()
    if isinstance(value, (bytes, bytearray)):
        return bytes(value)
    if isinstance(value, dict):
        payload = value.get("bytes")
        if isinstance(payload, memoryview):
            payload = payload.tobytes()
        if isinstance(payload, (bytes, bytearray)):
            return bytes(payload)
        path_value = value.get("path")
        if path_value:
            candidates = [Path(path_value)]
            if source_dir is not None:
                candidates.append(Path(source_dir) / path_value)
            for candidate in candidates:
                if candidate.is_file():
                    return candidate.read_bytes()
    if isinstance(value, str):
        candidates = [Path(value)]
        if source_dir is not None:
            candidates.append(Path(source_dir) / value)
        for candidate in candidates:
            if candidate.is_file():
                return candidate.read_bytes()
    raise TypeError(f"Unsupported image cell: {type(value)}")


demo_records_by_order = {}
demo_file_manifest = []

for rel_path, file_group in tqdm(
    demo_selection.groupby("hf_source_file", sort=True),
    total=len(demo_source_files),
    desc="Reconstructing MMMU demos",
):
    parquet_path = demo_snapshot / str(rel_path)
    demo_file_manifest.append({
        "relative_path": str(rel_path),
        "size_bytes": parquet_path.stat().st_size,
        "sha256": sha256_file(parquet_path),
    })
    parquet_file = pq.ParquetFile(parquet_path)

    for row_group_value, rg_group in file_group.groupby("hf_row_group_idx", sort=True):
        require(pd.notna(row_group_value), f"Missing row-group locator in {rel_path}")
        row_group_idx = int(row_group_value)
        require(0 <= row_group_idx < parquet_file.num_row_groups, f"Bad row group in {rel_path}")
        table = parquet_file.read_row_group(row_group_idx)
        require("id" in table.column_names, f"No id column in {rel_path}")
        row_group_ids = [str(x) for x in table.column("id").to_pylist()]

        for csv_index, meta in rg_group.iterrows():
            sample_id = str(meta["question_id"])
            local_idx = int(meta["hf_local_row_in_group"])
            if not (0 <= local_idx < table.num_rows) or row_group_ids[local_idx] != sample_id:
                matches = [i for i, value in enumerate(row_group_ids) if value == sample_id]
                require(len(matches) == 1, f"Could not resolve exact demo ID {sample_id} in {rel_path}")
                local_idx = matches[0]

            row = table.slice(local_idx, 1).to_pylist()[0]
            require(str(row["id"]) == sample_id, f"Resolved wrong source row for {sample_id}")
            require(str(row["question"]) == str(meta["question"]), f"Question mismatch for {sample_id}")

            expected_image_hashes = json.loads(str(meta["image_sha256_json"]))
            observed_image_hashes = {}
            observed_total_bytes = 0
            for image_col in IMAGE_COLS:
                if image_col not in row or row[image_col] is None:
                    continue
                payload = bytes_from_image_cell(row[image_col], parquet_path.parent)
                if payload is None:
                    continue
                observed_image_hashes[image_col] = sha256_bytes(payload)
                observed_total_bytes += len(payload)

            require(
                observed_image_hashes == expected_image_hashes,
                f"Source image SHA-256 mismatch for {sample_id}: "
                f"observed={observed_image_hashes}, expected={expected_image_hashes}",
            )
            require(
                observed_total_bytes == int(meta["hf_image_bytes"]),
                f"Source image byte-count mismatch for {sample_id}",
            )

            order1 = int(csv_index) + 1
            require(order1 not in demo_records_by_order, f"Duplicate demo order {order1}")
            demo_records_by_order[order1] = {
                "corpus": "demonstration",
                "sample_id": sample_id,
                "sample_order": order1,
                "subject": str(meta["subject"]),
                "source_split": str(meta["source_split"]),
                "id_prefix_split": sample_id.split("_", 1)[0],
                "source_repo": DEMO_REPO,
                "source_revision": DEMO_REVISION,
                "source_locator": (
                    f"{rel_path}#row_group={row_group_idx};local_row={local_idx};"
                    f"global_split_offset={int(meta['hf_global_split_offset'])}"
                ),
                "source_row_index": int(meta["hf_global_split_offset"]),
                "question": str(row["question"]),
                "source_row": row,
            }

demo_records = [demo_records_by_order[i] for i in range(1, DEMO_EXPECTED_ROWS + 1)]
require([r["sample_id"] for r in demo_records] == demo_ids, "Demo reconstruction order mismatch")
require(len(demo_records) == DEMO_EXPECTED_ROWS, "Demo reconstruction row-count mismatch")

atomic_json(OUTPUT_ROOT / "demonstration_source_provenance.json", {
    "captured_utc": utc_now(),
    "repo": DEMO_REPO,
    "revision": DEMO_REVISION,
    "selected_rows": len(demo_records),
    "selected_ids_canonical_sha256": canonical_id_sha256([r["sample_id"] for r in demo_records]),
    "source_files": demo_file_manifest,
    "source_validation": "exact ID + exact question + encoded-image bytes + encoded-image SHA-256",
})
print("MMMU exact-ID reconstruction and source-byte audit: PASS", len(demo_records))


Fetching 61 files:   0%|          | 0/61 [00:00<?, ?it/s]

Agriculture/test-00000-of-00002.parquet:   0%|          | 0.00/496M [00:00<?, ?B/s]

Architecture_and_Engineering/test-00000-(…):   0%|          | 0.00/15.9M [00:00<?, ?B/s]

Agriculture/dev-00000-of-00001.parquet:   0%|          | 0.00/22.1M [00:00<?, ?B/s]

Agriculture/validation-00000-of-00001.pa(…):   0%|          | 0.00/119M [00:00<?, ?B/s]

...

Public_Health/test-00000-of-00001.parque(…):   0%|          | 0.00/31.7M [00:00<?, ?B/s]

Sociology/dev-00000-of-00001.parquet:   0%|          | 0.00/3.78M [00:00<?, ?B/s]

Sociology/test-00000-of-00001.parquet:   0%|          | 0.00/144M [00:00<?, ?B/s]

Reconstructing MMMU demos:   0%|          | 0/61 [00:00<?, ?it/s]

MMMU exact-ID reconstruction and source-byte audit: PASS 405


In [6]:
# 5. Construct question-only text inputs and one independent item per physical image field
def normalize_question_for_clip(question):
    without_markers = IMAGE_REF_RE.sub(" ", str(question))
    return WHITESPACE_RE.sub(" ", without_markers).strip()


def decode_official_image(value, source_dir=None):
    encoded = bytes_from_image_cell(value, source_dir)
    require(encoded is not None and len(encoded) > 0, "Empty source image")
    with Image.open(io.BytesIO(encoded)) as opened:
        opened.load()
        oriented = ImageOps.exif_transpose(opened)
        original_mode = oriented.mode
        rgb = oriented.convert("RGB")
        rgb.load()
    return encoded, rgb, original_mode


def decoded_rgb_sha256(rgb_image):
    header = f"RGB|{rgb_image.width}|{rgb_image.height}|".encode("ascii")
    return sha256_bytes(header + rgb_image.tobytes())


def prepare_corpus(records):
    question_items = []
    image_items = []
    for record in tqdm(records, desc=f"Preparing {records[0]['corpus']}"):
        row = record["source_row"]
        raw_question = str(record["question"])
        marker_indices = [int(x) for x in IMAGE_REF_RE.findall(raw_question)]
        marker_cols = [f"image_{idx}" for idx in marker_indices]
        available_cols = []
        for image_col in IMAGE_COLS:
            if image_col not in row or row[image_col] is None:
                continue
            payload = bytes_from_image_cell(row[image_col])
            if payload:
                available_cols.append(image_col)

        require(available_cols, f"No source image for {record['sample_id']}")
        missing_marker_images = sorted(set(marker_cols) - set(available_cols))
        require(
            not missing_marker_images,
            f"Question {record['sample_id']} references absent images {missing_marker_images}",
        )

        embedding_text = normalize_question_for_clip(raw_question)
        has_text = bool(embedding_text)
        question_items.append({
            "corpus": record["corpus"],
            "sample_id": record["sample_id"],
            "sample_order": record["sample_order"],
            "subject": record["subject"],
            "source_split": record["source_split"],
            "id_prefix_split": record["id_prefix_split"],
            "source_repo": record["source_repo"],
            "source_revision": record["source_revision"],
            "source_locator": record["source_locator"],
            "source_row_index": record["source_row_index"],
            "question_raw": raw_question,
            "question_embedding_text": embedding_text,
            "question_raw_sha256": sha256_bytes(raw_question.encode("utf-8")),
            "question_embedding_text_sha256": sha256_bytes(embedding_text.encode("utf-8")),
            "question_has_text_after_marker_removal": has_text,
            "text_similarity_eligible": has_text,
            "empty_text_policy": EMPTY_TEXT_POLICY,
            "image_columns_json": json.dumps(available_cols),
            "image_marker_sequence_json": json.dumps(marker_cols),
            "n_unique_images": len(available_cols),
            "n_image_occurrences": len(marker_cols),
            "text_normalization_policy": TEXT_NORMALIZATION_POLICY,
        })

        for image_col in available_cols:
            encoded, rgb, original_mode = decode_official_image(row[image_col])
            marker_positions = [i + 1 for i, col in enumerate(marker_cols) if col == image_col]
            image_items.append({
                "corpus": record["corpus"],
                "sample_id": record["sample_id"],
                "sample_order": record["sample_order"],
                "subject": record["subject"],
                "source_split": record["source_split"],
                "id_prefix_split": record["id_prefix_split"],
                "source_repo": record["source_repo"],
                "source_revision": record["source_revision"],
                "source_locator": record["source_locator"],
                "source_row_index": record["source_row_index"],
                "image_col": image_col,
                "image_index": int(image_col.split("_")[1]),
                "is_referenced": bool(marker_positions),
                "marker_occurrence_count": len(marker_positions),
                "marker_positions_json": json.dumps(marker_positions),
                "source_encoded_bytes": len(encoded),
                "source_encoded_sha256": sha256_bytes(encoded),
                "decoded_rgb_sha256": decoded_rgb_sha256(rgb),
                "width": rgb.width,
                "height": rgb.height,
                "source_oriented_mode": original_mode,
                "embedding_input_mode": "RGB",
                "image_preprocess_policy": IMAGE_PREPROCESS_POLICY,
                "_pil": rgb,
            })

    question_items.sort(key=lambda x: x["sample_order"])
    image_items.sort(key=lambda x: (x["sample_order"], x["image_index"]))
    return question_items, image_items


query_question_items, query_image_items = prepare_corpus(query_records)
demo_question_items, demo_image_items = prepare_corpus(demo_records)

require(len(query_question_items) == 300, "Expected 300 query question items")
require(len(demo_question_items) == 405, "Expected 405 demo question items")
require(len(demo_image_items) == 405, "Frozen demo corpus must contain 405 physical images")
require([x["sample_id"] for x in query_question_items] == query_ids, "Query order changed")
require([x["sample_id"] for x in demo_question_items] == demo_ids, "Demo order changed")

input_construction_audit = {
    "captured_utc": utc_now(),
    "text_normalization_policy": TEXT_NORMALIZATION_POLICY,
    "empty_text_policy": EMPTY_TEXT_POLICY,
    "image_preprocess_policy": IMAGE_PREPROCESS_POLICY,
    "options_embedded": False,
    "cot_embedded": False,
    "answers_embedded": False,
    "explanations_embedded": False,
    "duplicate_marker_policy": "one embedding per physical image column; marker occurrences archived",
    "multi_image_aggregation": None,
    "query_questions": len(query_question_items),
    "query_empty_text_rows": sum(not x["question_has_text_after_marker_removal"] for x in query_question_items),
    "query_empty_text_ids": [x["sample_id"] for x in query_question_items if not x["question_has_text_after_marker_removal"]],
    "query_physical_images": len(query_image_items),
    "query_image_count_distribution": dict(sorted(Counter(x["n_unique_images"] for x in query_question_items).items())),
    "query_marker_occurrence_distribution": dict(sorted(Counter(x["n_image_occurrences"] for x in query_question_items).items())),
    "demo_questions": len(demo_question_items),
    "demo_empty_text_rows": sum(not x["question_has_text_after_marker_removal"] for x in demo_question_items),
    "demo_empty_text_ids": [x["sample_id"] for x in demo_question_items if not x["question_has_text_after_marker_removal"]],
    "demo_physical_images": len(demo_image_items),
    "demo_image_count_distribution": dict(sorted(Counter(x["n_unique_images"] for x in demo_question_items).items())),
    "demo_marker_occurrence_distribution": dict(sorted(Counter(x["n_image_occurrences"] for x in demo_question_items).items())),
}
atomic_json(OUTPUT_ROOT / "embedding_input_construction_audit.json", input_construction_audit)
print(json.dumps(input_construction_audit, indent=2))


Preparing query:   0%|          | 0/300 [00:00<?, ?it/s]

Preparing demonstration:   0%|          | 0/405 [00:00<?, ?it/s]

{
  "captured_utc": "2026-09-05T09:56:04.384441+00:00",
  "text_normalization_policy": "remove literal <image n> markers; Unicode preserved; collapse all whitespace; strip",
  "empty_text_policy": "if marker removal leaves no text: store a 768-dimensional float32 zero vector; set question_has_text_after_marker_removal=false and text_similarity_eligible=false",
  "image_preprocess_policy": "decode official source bytes; apply EXIF orientation; convert RGB; CLIPProcessor at pinned model revision",
  "options_embedded": false,
  "cot_embedded": false,
  "answers_embedded": false,
  "explanations_embedded": false,
  "duplicate_marker_policy": "one embedding per physical image column; marker occurrences archived",
  "multi_image_aggregation": null,
  "query_questions": 300,
  "query_empty_text_rows": 1,
  "query_empty_text_ids": [
    "test_Music_299"
  ],
  "query_physical_images": 345,
  "query_image_count_distribution": {
    "1": 278,
    "2": 12,
    "3": 1,
    "4": 5,
    "5": 4
  },

In [7]:
# 6. Download and verify the exact CLIP ViT-L/14 snapshot, then load in float32
model_snapshot = Path(snapshot_download(
    repo_id=CLIP_REPO,
    repo_type="model",
    revision=CLIP_REVISION,
    allow_patterns=[
        "config.json", "preprocessor_config.json", "tokenizer.json",
        "tokenizer_config.json", "special_tokens_map.json", "vocab.json",
        "merges.txt", "model.safetensors",
    ],
    local_dir=str(MODEL_CACHE),
))
model_weights = model_snapshot / "model.safetensors"
require(model_weights.is_file(), "Pinned CLIP model.safetensors was not downloaded")
observed_model_sha = sha256_file(model_weights)
require(
    observed_model_sha == CLIP_EXPECTED_MODEL_SHA256,
    f"CLIP model weight SHA mismatch: {observed_model_sha}",
)

processor = CLIPProcessor.from_pretrained(str(model_snapshot), local_files_only=True)
model = CLIPModel.from_pretrained(
    str(model_snapshot),
    local_files_only=True,
    use_safetensors=True,
    torch_dtype=torch.float32,
).to(DEVICE)
model.eval()

clip_context = int(processor.tokenizer.model_max_length)
clip_dim = int(model.config.projection_dim)
require(clip_context == CLIP_EXPECTED_CONTEXT, f"Unexpected CLIP context length: {clip_context}")
require(clip_dim == CLIP_EXPECTED_DIM, f"Unexpected CLIP projection dimension: {clip_dim}")

model_file_manifest = []
for path in sorted(model_snapshot.iterdir()):
    if path.is_file() and not path.name.startswith("."):
        model_file_manifest.append({
            "filename": path.name,
            "size_bytes": path.stat().st_size,
            "sha256": sha256_file(path),
        })

model_audit = {
    "captured_utc": utc_now(),
    "repo": CLIP_REPO,
    "revision": CLIP_REVISION,
    "model_safetensors_sha256": observed_model_sha,
    "projection_dim": clip_dim,
    "text_context_length": clip_context,
    "device": str(DEVICE),
    "inference_dtype": "float32",
    "vector_norm_policy": (
        "unit-L2 for present text/image modalities; zero vector for absent text"
    ),
    "files": model_file_manifest,
}
atomic_json(OUTPUT_ROOT / "embedding_model_provenance.json", model_audit)
print(json.dumps(model_audit, indent=2)[:8000])


Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/905 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


{
  "captured_utc": "2026-09-05T09:56:25.145325+00:00",
  "repo": "openai/clip-vit-large-patch14",
  "revision": "32bd64288804d66eefd0ccbe215aa642df71cc41",
  "model_safetensors_sha256": "a2bf730a0c7debf160f7a6b50b3aaf3703e7e88ac73de7a314903141db026dcb",
  "projection_dim": 768,
  "text_context_length": 77,
  "device": "cuda:0",
  "inference_dtype": "float32",
  "vector_norm_policy": "unit-L2 for present text/image modalities; zero vector for absent text",
  "files": [
    {
      "filename": "config.json",
      "size_bytes": 4519,
      "sha256": "8a09b467700c58138c29d53c605b34ebc69beaadd13274a8a2af8ad2c2f4032a"
    },
    {
      "filename": "merges.txt",
      "size_bytes": 524619,
      "sha256": "9fd691f7c8039210e0fced15865466c65820d09b63988b0174bfe25de299051a"
    },
    {
      "filename": "model.safetensors",
      "size_bytes": 1710540580,
      "sha256": "a2bf730a0c7debf160f7a6b50b3aaf3703e7e88ac73de7a314903141db026dcb"
    },
    {
      "filename": "preprocessor_config.jso

In [8]:
# 7. Embed question stems only (no options, answer, explanation, or CoT)
def embed_questions(items, batch_size=TEXT_BATCH_SIZE):
    texts = [x["question_embedding_text"] for x in items]
    token_counts_with_special = [
        len(processor.tokenizer(text, add_special_tokens=True, truncation=False)["input_ids"])
        for text in texts
    ]
    token_counts_content = [
        len(processor.tokenizer(text, add_special_tokens=False, truncation=False)["input_ids"])
        for text in texts
    ]
    eligible_indices = [i for i, text in enumerate(texts) if bool(text)]
    embeddings = np.zeros((len(items), clip_dim), dtype=np.float32)
    raw_norms_by_index = [None] * len(items)

    with torch.inference_mode():
        for start in tqdm(range(0, len(eligible_indices), batch_size), desc="CLIP text embeddings"):
            batch_indices = eligible_indices[start:start + batch_size]
            batch_texts = [texts[i] for i in batch_indices]
            encoded = processor(
                text=batch_texts,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=clip_context,
            )
            encoded = {key: value.to(DEVICE) for key, value in encoded.items()}
            features = model.get_text_features(**encoded).float()
            raw_norms = torch.linalg.vector_norm(features, dim=1)
            require(torch.isfinite(features).all().item(), "Non-finite CLIP text feature")
            require((raw_norms > 0).all().item(), "Zero CLIP text feature")
            normalized = features / raw_norms.unsqueeze(1)
            batch_embeddings = normalized.cpu().numpy().astype(np.float32, copy=False)
            batch_raw_norms = raw_norms.cpu().numpy().astype(float).tolist()
            for local_idx, item_idx in enumerate(batch_indices):
                embeddings[item_idx] = batch_embeddings[local_idx]
                raw_norms_by_index[item_idx] = float(batch_raw_norms[local_idx])

    require(embeddings.shape == (len(items), clip_dim), "Text embedding shape mismatch")
    require(np.isfinite(embeddings).all(), "Non-finite saved text embedding")
    observed_norms = np.linalg.norm(embeddings, axis=1)
    expected_norms = np.asarray([1.0 if text else 0.0 for text in texts], dtype=np.float32)
    require(np.max(np.abs(observed_norms - expected_norms)) < 2e-5, "Text norm/missingness check failed")

    metadata = []
    for item, token_with_special, token_content, raw_norm in zip(
        items, token_counts_with_special, token_counts_content, raw_norms_by_index
    ):
        has_text = bool(item["question_has_text_after_marker_removal"])
        row = dict(item)
        row.update({
            "clip_token_count_before_truncation": int(token_with_special),
            "clip_content_token_count_before_truncation": int(token_content),
            "clip_context_length": clip_context,
            "was_truncated": bool(token_with_special > clip_context),
            "raw_feature_l2_norm": None if raw_norm is None else float(raw_norm),
            "embedding_model": CLIP_REPO,
            "embedding_revision": CLIP_REVISION,
            "embedding_dim": clip_dim,
            "embedding_dtype": "float32",
            "embedding_l2_normalized": has_text,
            "embedding_is_zero_missing_modality": not has_text,
        })
        metadata.append(row)
    return metadata, embeddings


query_question_meta, query_question_embeddings = embed_questions(query_question_items)
demo_question_meta, demo_question_embeddings = embed_questions(demo_question_items)

print("Query text truncations:", sum(x["was_truncated"] for x in query_question_meta), "/", len(query_question_meta))
print("Demo text truncations :", sum(x["was_truncated"] for x in demo_question_meta), "/", len(demo_question_meta))


Token indices sequence length is longer than the specified maximum sequence length for this model (196 > 77). Running this sequence through the model will result in indexing errors


CLIP text embeddings:   0%|          | 0/5 [00:00<?, ?it/s]

CLIP text embeddings:   0%|          | 0/7 [00:00<?, ?it/s]

Query text truncations: 54 / 300
Demo text truncations : 97 / 405


In [9]:
# 8. Embed each physical image independently; no hidden multi-image aggregation
def embed_images(items, batch_size=IMAGE_BATCH_SIZE):
    all_embeddings = []
    all_raw_norms = []

    with torch.inference_mode():
        for start in tqdm(range(0, len(items), batch_size), desc="CLIP image embeddings"):
            batch_items = items[start:start + batch_size]
            encoded = processor(
                images=[x["_pil"] for x in batch_items],
                return_tensors="pt",
            )
            pixel_values = encoded["pixel_values"].to(DEVICE, dtype=torch.float32)
            features = model.get_image_features(pixel_values=pixel_values).float()
            raw_norms = torch.linalg.vector_norm(features, dim=1)
            require(torch.isfinite(features).all().item(), "Non-finite CLIP image feature")
            require((raw_norms > 0).all().item(), "Zero CLIP image feature")
            normalized = features / raw_norms.unsqueeze(1)
            all_embeddings.append(normalized.cpu().numpy().astype(np.float32, copy=False))
            all_raw_norms.extend(raw_norms.cpu().numpy().astype(float).tolist())

    embeddings = np.concatenate(all_embeddings, axis=0)
    require(embeddings.shape == (len(items), clip_dim), "Image embedding shape mismatch")
    require(np.isfinite(embeddings).all(), "Non-finite saved image embedding")
    require(np.max(np.abs(np.linalg.norm(embeddings, axis=1) - 1.0)) < 2e-5, "Image L2 check failed")

    metadata = []
    for item, raw_norm in zip(items, all_raw_norms):
        row = {key: value for key, value in item.items() if key != "_pil"}
        row.update({
            "raw_feature_l2_norm": float(raw_norm),
            "embedding_model": CLIP_REPO,
            "embedding_revision": CLIP_REVISION,
            "embedding_dim": clip_dim,
            "embedding_dtype": "float32",
            "embedding_l2_normalized": True,
        })
        metadata.append(row)
    return metadata, embeddings


query_image_meta, query_image_embeddings = embed_images(query_image_items)
demo_image_meta, demo_image_embeddings = embed_images(demo_image_items)

# Release decoded PIL objects and model memory before serialization.
for item in query_image_items + demo_image_items:
    item.pop("_pil", None)
del model
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Query physical images:", len(query_image_meta))
print("Demo physical images :", len(demo_image_meta))


CLIP image embeddings:   0%|          | 0/22 [00:00<?, ?it/s]

CLIP image embeddings:   0%|          | 0/26 [00:00<?, ?it/s]

Query physical images: 345
Demo physical images : 405


In [10]:
# 9. Write four canonical Parquet datasets plus retrieval-ready NPZ matrices
SCHEMA_METADATA = {
    b"protocol_version": PROTOCOL_VERSION.encode("utf-8"),
    b"embedding_model": CLIP_REPO.encode("utf-8"),
    b"embedding_revision": CLIP_REVISION.encode("utf-8"),
    b"embedding_dim": str(clip_dim).encode("ascii"),
    b"embedding_dtype": b"float32",
    b"embedding_norm_policy": b"unit-L2 for present modalities; zero vector for absent text",
    b"options_embedded": b"false",
    b"cot_embedded": b"false",
    b"multi_image_aggregation": b"none",
}


def write_embedding_parquet(path, metadata_rows, embeddings):
    path = Path(path)
    require(len(metadata_rows) == embeddings.shape[0], f"Row mismatch for {path.name}")
    require(embeddings.shape[1] == clip_dim, f"Dimension mismatch for {path.name}")
    meta_table = pa.Table.from_pylist(metadata_rows)
    flat = pa.array(embeddings.reshape(-1), type=pa.float32())
    embedding_column = pa.FixedSizeListArray.from_arrays(flat, clip_dim)
    table = meta_table.append_column("embedding", embedding_column)
    prior_metadata = dict(table.schema.metadata or {})
    table = table.replace_schema_metadata({**prior_metadata, **SCHEMA_METADATA})
    tmp = path.with_suffix(path.suffix + ".tmp")
    pq.write_table(table, tmp, compression="zstd", use_dictionary=True)
    os.replace(tmp, path)


def write_npz(path, metadata_rows, embeddings, modality):
    path = Path(path)
    payload = {
        "sample_id": np.asarray([x["sample_id"] for x in metadata_rows], dtype=str),
        "sample_order": np.asarray([x["sample_order"] for x in metadata_rows], dtype=np.int32),
        "subject": np.asarray([x["subject"] for x in metadata_rows], dtype=str),
        "embedding": embeddings.astype(np.float32, copy=False),
        "embedding_model": np.asarray(CLIP_REPO),
        "embedding_revision": np.asarray(CLIP_REVISION),
        "l2_normalized": np.asarray(
            [bool(x["embedding_l2_normalized"]) for x in metadata_rows], dtype=bool
        ),
    }
    if modality == "question":
        payload["text_similarity_eligible"] = np.asarray(
            [bool(x["text_similarity_eligible"]) for x in metadata_rows], dtype=bool
        )
        payload["embedding_is_zero_missing_modality"] = np.asarray(
            [bool(x["embedding_is_zero_missing_modality"]) for x in metadata_rows], dtype=bool
        )
    if modality == "image":
        payload["image_col"] = np.asarray([x["image_col"] for x in metadata_rows], dtype=str)
        payload["image_index"] = np.asarray([x["image_index"] for x in metadata_rows], dtype=np.int8)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with tmp.open("wb") as handle:
        np.savez_compressed(handle, **payload)
    os.replace(tmp, path)


artifacts = {
    "query_question_parquet": OUTPUT_ROOT / "query_question_embeddings.parquet",
    "query_image_parquet": OUTPUT_ROOT / "query_image_embeddings.parquet",
    "demo_question_parquet": OUTPUT_ROOT / "demonstration_question_embeddings.parquet",
    "demo_image_parquet": OUTPUT_ROOT / "demonstration_image_embeddings.parquet",
    "query_question_npz": OUTPUT_ROOT / "query_question_embeddings.npz",
    "query_image_npz": OUTPUT_ROOT / "query_image_embeddings.npz",
    "demo_question_npz": OUTPUT_ROOT / "demonstration_question_embeddings.npz",
    "demo_image_npz": OUTPUT_ROOT / "demonstration_image_embeddings.npz",
}

write_embedding_parquet(artifacts["query_question_parquet"], query_question_meta, query_question_embeddings)
write_embedding_parquet(artifacts["query_image_parquet"], query_image_meta, query_image_embeddings)
write_embedding_parquet(artifacts["demo_question_parquet"], demo_question_meta, demo_question_embeddings)
write_embedding_parquet(artifacts["demo_image_parquet"], demo_image_meta, demo_image_embeddings)

write_npz(artifacts["query_question_npz"], query_question_meta, query_question_embeddings, "question")
write_npz(artifacts["query_image_npz"], query_image_meta, query_image_embeddings, "image")
write_npz(artifacts["demo_question_npz"], demo_question_meta, demo_question_embeddings, "question")
write_npz(artifacts["demo_image_npz"], demo_image_meta, demo_image_embeddings, "image")

# Human-readable metadata tables contain no vector columns.
pd.DataFrame(query_question_meta).to_csv(OUTPUT_ROOT / "query_question_metadata.csv", index=False)
pd.DataFrame(query_image_meta).to_csv(OUTPUT_ROOT / "query_image_metadata.csv", index=False)
pd.DataFrame(demo_question_meta).to_csv(OUTPUT_ROOT / "demonstration_question_metadata.csv", index=False)
pd.DataFrame(demo_image_meta).to_csv(OUTPUT_ROOT / "demonstration_image_metadata.csv", index=False)

print("Canonical embedding files written.")


Canonical embedding files written.


In [11]:
# 10. Re-open every canonical dataset and execute the final paper-run validation gate
FORBIDDEN_OUTPUT_COLUMNS = {
    "options", "options_json", "answer", "gold_answer", "explanation",
    "original_explanation", "generated_cot", "cot", "call1_raw_response",
}


def validate_parquet(path, expected_rows, expected_ids, allow_repeated_ids, modality):
    table = pq.read_table(path)
    require(table.num_rows == expected_rows, f"{path.name}: row-count mismatch")
    require("embedding" in table.column_names, f"{path.name}: embedding missing")
    require(not (FORBIDDEN_OUTPUT_COLUMNS & set(table.column_names)), f"{path.name}: forbidden field leaked")
    vectors = np.asarray(table.column("embedding").to_pylist(), dtype=np.float32)
    ids = [str(x) for x in table.column("sample_id").to_pylist()]
    require(vectors.shape == (expected_rows, clip_dim), f"{path.name}: shape mismatch")
    require(np.isfinite(vectors).all(), f"{path.name}: non-finite vector")
    observed_norms = np.linalg.norm(vectors, axis=1)
    if modality == "question":
        require("text_similarity_eligible" in table.column_names, f"{path.name}: missing text flag")
        eligible = np.asarray(table.column("text_similarity_eligible").to_pylist(), dtype=bool)
        expected_norms = eligible.astype(np.float32)
    else:
        eligible = np.ones(expected_rows, dtype=bool)
        expected_norms = np.ones(expected_rows, dtype=np.float32)
    norm_error = np.abs(observed_norms - expected_norms)
    require(np.max(norm_error) < 2e-5, f"{path.name}: norm/missingness mismatch")
    if allow_repeated_ids:
        require(set(ids) == set(expected_ids), f"{path.name}: parent-ID coverage mismatch")
    else:
        require(ids == expected_ids, f"{path.name}: ID order mismatch")
    field = table.schema.field("embedding")
    require(pa.types.is_fixed_size_list(field.type), f"{path.name}: embedding is not fixed-size list")
    require(field.type.list_size == clip_dim, f"{path.name}: list dimension mismatch")
    require(field.type.value_type == pa.float32(), f"{path.name}: vector dtype is not float32")
    schema_meta = table.schema.metadata or {}
    require(schema_meta.get(b"embedding_revision") == CLIP_REVISION.encode(), f"{path.name}: revision metadata mismatch")
    require(schema_meta.get(b"multi_image_aggregation") == b"none", f"{path.name}: aggregation metadata mismatch")
    return {
        "rows": table.num_rows,
        "columns": table.column_names,
        "embedding_shape": list(vectors.shape),
        "max_norm_policy_error": float(np.max(norm_error)),
        "zero_vector_missing_modality_rows": int((~eligible).sum()),
        "sha256": sha256_file(path),
        "size_bytes": path.stat().st_size,
    }


validation = {
    "query_question_embeddings.parquet": validate_parquet(
        artifacts["query_question_parquet"], len(query_question_meta), query_ids, False, "question"
    ),
    "query_image_embeddings.parquet": validate_parquet(
        artifacts["query_image_parquet"], len(query_image_meta), query_ids, True, "image"
    ),
    "demonstration_question_embeddings.parquet": validate_parquet(
        artifacts["demo_question_parquet"], len(demo_question_meta), demo_ids, False, "question"
    ),
    "demonstration_image_embeddings.parquet": validate_parquet(
        artifacts["demo_image_parquet"], len(demo_image_meta), demo_ids, True, "image"
    ),
}

for name, path in artifacts.items():
    if path.suffix == ".npz":
        with np.load(path, allow_pickle=False) as loaded:
            require(loaded["embedding"].dtype == np.float32, f"{path.name}: NPZ dtype mismatch")
            require(loaded["embedding"].shape[1] == clip_dim, f"{path.name}: NPZ dim mismatch")
            observed_norms = np.linalg.norm(loaded["embedding"], axis=1)
            expected_norms = loaded["l2_normalized"].astype(np.float32)
            require(np.max(np.abs(observed_norms - expected_norms)) < 2e-5, f"{path.name}: NPZ norm mismatch")
        validation[path.name] = {
            "sha256": sha256_file(path),
            "size_bytes": path.stat().st_size,
        }

embedding_manifest = {
    "status": "PASS",
    "created_utc": utc_now(),
    "protocol_version": PROTOCOL_VERSION,
    "query_source": {
        "repo": QUERY_REPO,
        "setting": QUERY_SUBDIR,
        "split": QUERY_SPLIT,
        "revision": QUERY_REVISION,
        "selected_rows": len(query_ids),
        "selected_ids_canonical_sha256": canonical_id_sha256(query_ids),
    },
    "demonstration_source": {
        "repo": DEMO_REPO,
        "revision": DEMO_REVISION,
        "selected_rows": len(demo_ids),
        "selection_csv_sha256": sha256_file(DEMO_CSV_PATH),
        "selected_ids_canonical_sha256": canonical_id_sha256(demo_ids),
    },
    "embedding_model": {
        "repo": CLIP_REPO,
        "revision": CLIP_REVISION,
        "model_safetensors_sha256": observed_model_sha,
        "dimension": clip_dim,
        "text_context_length": clip_context,
        "output_dtype": "float32",
        "vector_norm_policy": "unit-L2 for present modalities; zero vector for absent text",
    },
    "semantic_input": {
        "text": "question stem only",
        "text_normalization_policy": TEXT_NORMALIZATION_POLICY,
        "empty_text_policy": EMPTY_TEXT_POLICY,
        "image": "each physical image field independently",
        "image_preprocess_policy": IMAGE_PREPROCESS_POLICY,
        "options_embedded": False,
        "answers_embedded": False,
        "explanations_embedded": False,
        "cot_embedded": False,
        "multi_image_aggregation": None,
    },
    "text_truncation_audit": {
        "query_truncated": int(sum(x["was_truncated"] for x in query_question_meta)),
        "query_total": len(query_question_meta),
        "demo_truncated": int(sum(x["was_truncated"] for x in demo_question_meta)),
        "demo_total": len(demo_question_meta),
    },
    "text_missingness_audit": {
        "query_empty_text_rows": int(sum(not x["text_similarity_eligible"] for x in query_question_meta)),
        "query_empty_text_ids": [x["sample_id"] for x in query_question_meta if not x["text_similarity_eligible"]],
        "demo_empty_text_rows": int(sum(not x["text_similarity_eligible"] for x in demo_question_meta)),
        "demo_empty_text_ids": [x["sample_id"] for x in demo_question_meta if not x["text_similarity_eligible"]],
        "policy": EMPTY_TEXT_POLICY,
    },
    "artifacts": validation,
    "environment": ENVIRONMENT,
}
atomic_json(OUTPUT_ROOT / "embedding_manifest.json", embedding_manifest)
atomic_json(OUTPUT_ROOT / "FINAL_VALIDATION.json", {
    "status": "PASS",
    "validated_utc": utc_now(),
    "protocol_version": PROTOCOL_VERSION,
    "checks": [
        "exact frozen input hashes",
        "exact-ID reconstruction from pinned official datasets",
        "demo source question and encoded-image byte/hash equality",
        "no options/answers/explanations/CoT in canonical outputs",
        "one row per question and one row per physical image",
        "fixed-size 768-dimensional float32 vectors",
        "finite unit-L2 vectors for present modalities and zero vectors for absent text",
        "query and demonstration ordering preserved",
        "no multi-image aggregation",
    ],
})

readme = f"""# RICES CLIP embedding corpus

Status: **PASS**

Protocol: `{PROTOCOL_VERSION}`

## Canonical data

- `query_question_embeddings.parquet`: {len(query_question_meta)} rows
- `query_image_embeddings.parquet`: {len(query_image_meta)} rows
- `demonstration_question_embeddings.parquet`: {len(demo_question_meta)} rows
- `demonstration_image_embeddings.parquet`: {len(demo_image_meta)} rows

Every embedding is float32 and 768-dimensional. Present text/image modalities
are L2-normalized. If removing image markers leaves no question text, that
missing text modality is stored as a flagged zero vector. The matching `.npz`
files are optimized for later matrix multiplication. Metadata-only CSVs are
provided for inspection.

## Semantic exclusions

Options, gold labels, answers, source explanations, and generated CoT were not
embedded and are not present in the four canonical Parquet datasets.

## Multi-image policy

Every physical source image column is stored independently. No image vectors
were averaged or maximized. Marker order, occurrence count, and marker
positions are stored in metadata so a later retrieval experiment can declare
its aggregation rule explicitly.

## Later RICES scoring

For present modalities, cosine similarity is a dot product. A flagged
text-missing zero vector contributes exactly zero to `s(T_i,T_q)`. Do not sum
image and text scores until a frozen multi-image aggregation rule has been
declared for samples containing more than one image.
"""
(OUTPUT_ROOT / "README.md").write_text(readme, encoding="utf-8")

print("=" * 88)
print("FINAL RICES EMBEDDING CORPUS VALIDATION: PASS")
print("Output:", OUTPUT_ROOT)
for path in sorted(OUTPUT_ROOT.iterdir()):
    if path.is_file():
        print(f"- {path.name}: {path.stat().st_size / (1024**2):.2f} MiB")
print("=" * 88)


FINAL RICES EMBEDDING CORPUS VALIDATION: PASS
Output: /kaggle/working/RICES_MMMU_MMMUPro_CLIP_ViTL14_embeddings_v1_1
- FINAL_VALIDATION.json: 0.00 MiB
- README.md: 0.00 MiB
- demonstration_ids.csv: 0.01 MiB
- demonstration_image_embeddings.npz: 1.07 MiB
- demonstration_image_embeddings.parquet: 1.73 MiB
- demonstration_image_metadata.csv: 0.23 MiB
- demonstration_question_embeddings.npz: 1.04 MiB
- demonstration_question_embeddings.parquet: 1.76 MiB
- demonstration_question_metadata.csv: 0.46 MiB
- demonstration_source_provenance.json: 0.01 MiB
- embedding_input_construction_audit.json: 0.00 MiB
- embedding_manifest.json: 0.01 MiB
- embedding_model_provenance.json: 0.00 MiB
- environment.json: 0.00 MiB
- pip_freeze.txt: 0.02 MiB
- query_image_embeddings.npz: 0.95 MiB
- query_image_embeddings.parquet: 1.57 MiB
- query_image_metadata.csv: 0.17 MiB
- query_question_embeddings.npz: 0.82 MiB
- query_question_embeddings.parquet: 1.39 MiB
- query_question_metadata.csv: 0.33 MiB
- query_source

## Suggested Methods wording

> **RICES embedding construction.** The fixed 300-item MMMU-Pro query cohort
> and 405-item MMMU demonstration pool were reconstructed by exact ordered ID
> lookup from immutable source revisions. We encoded each question stem and
> each physical source image independently using CLIP ViT-L/14
> (`openai/clip-vit-large-patch14`, revision
> `32bd64288804d66eefd0ccbe215aa642df71cc41`). Text inputs excluded image
> placeholders, answer options, gold labels, source explanations, and generated
> chains of thought. Images were decoded from the official source bytes, EXIF
> orientation was applied, and inputs were converted to RGB before the pinned
> CLIP preprocessing pipeline. Present modalities were stored as
> L2-normalized 768-dimensional float32 vectors. When marker removal left no
> question text, the missing text modality was explicitly flagged and stored
> as a zero vector, making its later text-similarity contribution zero.
> Multi-image samples were retained as separate image vectors, with marker
> positions and occurrence counts archived; no unreported image-vector
> aggregation was applied during corpus construction.

### Important interpretation note

CLIP has a 77-token text context. The notebook therefore records each
pre-truncation token count and a `was_truncated` flag for every question. This
does not silently change the protocol: truncation is the defined behavior of
the pinned CLIP text encoder and its prevalence is made auditable.

For image-only questions such as `test_Music_299`, no artificial sentinel text
is introduced. The zero vector and explicit missing-text flags preserve the
absence of a textual modality without injecting arbitrary CLIP semantics.
